# **GPU Image Processing with PyCUDA**

Image processing is one of the most powerful applications of GPU computing because:
* Each pixel can be processed independently
* Massive parallelism fits GPU architecture perfectly
* Performance scales extremely well with image size

We will learn about:
* Compare CPU vs GPU (PyCUDA)
* Apply image processing operations
* Use a medical imaging concept (cancer dataset style images)
* Visualize performance monitoring

In [ ]:
# Setup PyCUDA (Colab / Linux)
!pip install pycuda

  Using cached pycuda-2026.1.tar.gz (1.7 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pytools-2026.1.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached siphash24-1.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (3.2 kB)
Using cached pytools-2026.1.1-py3-none-any.whl (99 kB)
Using cached siphash24-1.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (103 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for pycuda (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pycuda
Failed to build pycuda
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pycuda)


In [ ]:
#Cek GPU
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


# **GPU Image Processing with PyCUDA (Medical Imaging Case Study)**


In [ ]:
# PyCUDA Kernel
import pycuda.autoinit
import pycuda.driver as cuda
import numpy as np
from pycuda.compiler import SourceModule
import time

mod = SourceModule("""
__global__ void brightness(unsigned char *img, unsigned char *out, int n, int value)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    if (idx < n)
    {
        int pixel = img[idx] + value;
        if (pixel > 255) pixel = 255;
        out[idx] = pixel;
    }
}
""")

func = mod.get_function("brightness")

In [ ]:
# GPU Function

def gpu_brightness(img, value=50):
    n = img.size
    out = np.zeros_like(img)

    img_gpu = cuda.mem_alloc(img.nbytes)
    out_gpu = cuda.mem_alloc(out.nbytes)

    cuda.memcpy_htod(img_gpu, img)

    block_size = 256
    grid_size = (n + block_size - 1) // block_size

    start = cuda.Event()
    end = cuda.Event()

    start.record()

    func(img_gpu, out_gpu, np.int32(n), np.int32(value),
         block=(block_size,1,1),
         grid=(grid_size,1))

    end.record()
    end.synchronize()

    cuda.memcpy_dtoh(out, out_gpu)

    gpu_time = start.time_till(end) / 1000
    return out, gpu_time

In [ ]:
# Load Cancer-style Image (Simulation)
img_size = 4096 * 4096  # large medical image simulation
img = np.random.randint(0, 256, img_size).astype(np.uint8)

In [ ]:
# Benchmark CPU vs GPU

_, cpu_time = cpu_brightness(img, 50)
_, gpu_time = gpu_brightness(img, 50)

print("CPU Time:", cpu_time)
print("GPU Time:", gpu_time)